# 10. Graph-Based Topic Modeling via FAISS and Leiden Community Detection

This notebook implements a topology-based topic modeling pipeline using embeddings fine-tuned with PRISM (CoSENT loss). Since the embeddings are semantically aligned via cosine distance, we construct an exact k-NN graph using FAISS, followed by the Leiden algorithm (Constant Potts Model) to isolate highly cohesive semantic communities.

## 10.1 Environment Setup

In [ ]:
import importlib.util
import subprocess
import sys

def check_and_install(package_name, pip_name=None):
    if pip_name is None:
        pip_name = package_name
    if importlib.util.find_spec(package_name) is None:
        try:
            from google.colab import drive
            print(f"Installing {pip_name} in Colab...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])
        except ImportError:
            print(f"Warning: '{package_name}' is missing. Make sure to install '{pip_name}' in your local .venv")

check_and_install("faiss", "faiss-cpu")
check_and_install("igraph")
check_and_install("leidenalg")

In [ ]:
import os
import numpy as np
import pandas as pd
import faiss
import igraph as ig
import leidenalg as la
from tqdm.auto import tqdm
from pathlib import Path

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
    BASE_PATH = Path("/content/drive/Shareddrives/Minería/proyecto_horus/")
else:
    BASE_PATH = Path("../")

# Constants
EMBEDDINGS_PATH = BASE_PATH / "data/processed/prism_embeddings.npy"
METADATA_PATH = BASE_PATH / "data/processed/products_modeling.parquet"
OUTPUT_PATH = BASE_PATH / "data/processed/graph_topics.parquet"

K_NEIGHBORS = 15
MIN_SIMILARITY = 0.82
RESOLUTION_PARAMETER = 0.05

## 10.2 Data Loading & Preparation

We load the 1024D float32 PRISM embeddings and the corresponding dataset metadata. The embeddings are L2 normalized to ensure FAISS inner product search mathematically equates to cosine similarity.

In [ ]:
# Load Data
print(f"Loading metadata from {METADATA_PATH}...")
df_meta = pd.read_parquet(METADATA_PATH)

print(f"Loading embeddings from {EMBEDDINGS_PATH}...")
embeddings = np.load(EMBEDDINGS_PATH)

print(f"Embeddings shape: {embeddings.shape}, dtype: {embeddings.dtype}")
print(f"Metadata shape: {df_meta.shape}")

# L2 Normalize embeddings for Cosine Similarity (Inner Product)
print("Normalizing embeddings for Cosine Similarity...")
faiss.normalize_L2(embeddings)

## 10.3 FAISS Exact K-NN Graph Construction

We build an exact k-NN graph. Since the dataset is ~100k vectors, an exact index (`IndexFlatIP`) fits perfectly in RAM and is extremely fast on CPU. We query the top `K=15` neighbors.

In [ ]:
d = embeddings.shape[1]  # 1024
index = faiss.IndexFlatIP(d)

print("Adding vectors to FAISS index...")
index.add(embeddings)
print(f"Total vectors in index: {index.ntotal}")

print(f"Searching top {K_NEIGHBORS} nearest neighbors...")
distances, indices = index.search(embeddings, K_NEIGHBORS)

print(f"Search complete. Distances shape: {distances.shape}")

## 10.4 Strict Edge Filtering & igraph Initialization

We filter out connections where the cosine similarity is below `MIN_SIMILARITY`. We also exclude self-loops to prevent artificially inflating cluster density.

In [ ]:
edges = []
weights = []

print(f"Filtering edges (Threshold: >= {MIN_SIMILARITY}) and removing self-loops...")

num_nodes = embeddings.shape[0]

for i in tqdm(range(num_nodes), desc="Processing neighbors"):
    for j_idx, neighbor in enumerate(indices[i]):
        similarity = distances[i, j_idx]
        
        # Avoid self loops (i == neighbor)
        if i == neighbor:
            continue
            
        # Filter by threshold
        if similarity >= MIN_SIMILARITY:
            # Ensure undirected representation (source < target) to avoid duplicate edges
            source, target = sorted([i, neighbor])
            edges.append((source, target))
            weights.append(similarity)

# Remove duplicate edges natively created by Bidirectional links in KNN
edge_dict = {edge: w for edge, w in zip(edges, weights)}
unique_edges = list(edge_dict.keys())
unique_weights = list(edge_dict.values())

print(f"Total unique edges passing threshold: {len(unique_edges)}")

# Build undirected igraph
print("Building igraph object...")
G = ig.Graph(n=num_nodes, edges=unique_edges, directed=False)
G.es['weight'] = unique_weights

print(f"Graph initialized: {G.vcount()} nodes, {G.ecount()} edges.")

## 10.5 Leiden Community Detection

We run community detection using the Leiden algorithm with the Constant Potts Model (CPM). CPM resolves the resolution limit problem, making it ideal for large networks.

In [ ]:
print(f"Running Leiden CPM Community Detection (resolution = {RESOLUTION_PARAMETER})...")

# Run Leiden using CPMVertexPartition
partition = la.find_partition(
    G, 
    la.CPMVertexPartition, 
    weights=G.es['weight'],
    resolution_parameter=RESOLUTION_PARAMETER
)

print(f"Modularity: {partition.modularity:.4f}")
print(f"Total communities discovered: {len(partition)}")

# Extract cluster IDs
community_assignments = np.array(partition.membership)

# Identify isolated nodes (degree 0) or communities of size 1 and label them as -1 (Outlier)
community_sizes = np.bincount(community_assignments)
outlier_communities = np.where(community_sizes == 1)[0]
community_assignments[np.isin(community_assignments, outlier_communities)] = -1

df_meta['community_id'] = community_assignments

# Renumber clusters so they are contiguous starting from 0, with -1 still representing outliers
unique_valid_clusters = sorted(set(community_assignments[community_assignments != -1]))
cluster_mapping = {old_id: new_id for new_id, old_id in enumerate(unique_valid_clusters)}
cluster_mapping[-1] = -1

df_meta['community_id'] = df_meta['community_id'].map(cluster_mapping)

total_valid_clusters = len(unique_valid_clusters)
print(f"Total valid communities (size > 1): {total_valid_clusters}")
print(f"Total outliers (unconnected nodes): {(df_meta['community_id'] == -1).sum()}")

## 10.6 Aggregation & Cluster Profiling

We evaluate the top 10 largest communities, sampling titles to validate semantic cohesion.

In [ ]:
print("Top 10 Largest Communities:")
top_communities = df_meta[df_meta['community_id'] != -1]['community_id'].value_counts().head(10)

for cluster_id, size in top_communities.items():
    print(f"\n========================================")
    print(f"Community ID: {cluster_id} | Size: {size}")
    print(f"========================================")
    samples = df_meta[df_meta['community_id'] == cluster_id]['original_title'].sample(n=min(5, size), random_state=42)
    for sample in samples:
        print(f"- {sample}")

## 10.7 Export

Save the resulting dataset with community assignments.

In [ ]:
print(f"Exporting dataset with graph topics to {OUTPUT_PATH}...")
df_meta.to_parquet(OUTPUT_PATH)
print("Done!")